<a href="https://colab.research.google.com/github/tassid/relacoes_fisicas_ambiente/blob/main/codigos_python/projeto_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto 1 · Elementos meteorológicos
### Radiação solar, saldo de radiação e ETo no norte do Paraná: Maringá, Londrina e Apucarana

**Disciplina:** Relações Físicas do Ambiente Agrícola (PPGTCA/UTFPR) · **Entrega:** 09/11/2026 · **Peso:** 30%

O projeto reproduz dois artigos e depois os integra, para três locais:

```
     Fernandes et al. (2018)              Fietz & Fisch (2009)
  ┌──────────────────────────┐     ┌───────────────────────────────┐
  │ Tmax, Tmin  ──►  Qg      │ ──► │ Qg ──► Rn ──► ETo (Priestley- │
  │ (5 modelos empíricos)    │     │       (4 modelos)    Taylor)  │
  └──────────────────────────┘     └───────────────────────────────┘
```

> **Pergunta central:** se eu tiver apenas termômetro, quanto erro carrego até a ETo? E em que elo da cadeia esse erro nasce?

| Etapa | Conteúdo |
|---|---|
| 0 | Locais de estudo |
| 1 | Extração do BR-DWGD (Xavier et al.) e controle de qualidade |
| 2 | Variáveis astronômicas e balanço de radiação (FAO-56) |
| 3 | Modelos de Qg a partir da temperatura (Fernandes et al., 2018) |
| 4 | Modelos de Rn e Priestley-Taylor (Fietz & Fisch, 2009) |
| 5 | Integração: temperatura → Qg → Rn → ETo |

Calibração com os **anos ímpares** e validação com os **anos pares** (2001–2025).

> Nas equações, **Qg** é a irradiância solar global (no BR-DWGD, banda `Rs`).

## Configuração do ambiente
___

In [1]:
!pip install -q agrometeorologiapy geemap

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import agrometeorologiapy as amp

# ID do projeto do Google Cloud registrado no Earth Engine
ID_PROJETO = 'ppgtca-relacoes-fisicas'

# True: salva a base no Google Drive (recomendado no Colab, evita repetir a extração)
USAR_DRIVE = True

In [3]:
PASTA_RAIZ = '.'
if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PASTA_RAIZ = '/content/drive/MyDrive/RFAA_P1'
    except ImportError:
        print('Fora do Colab: os arquivos ficam na pasta atual.')

PASTA_DADOS = os.path.join(PASTA_RAIZ, 'dados')
PASTA_FIGURAS = os.path.join(PASTA_RAIZ, 'figuras')
PASTA_RESULTADOS = os.path.join(PASTA_RAIZ, 'resultados')
for p in (PASTA_DADOS, PASTA_FIGURAS, PASTA_RESULTADOS):
    os.makedirs(p, exist_ok=True)
print('Arquivos em:', os.path.abspath(PASTA_RAIZ))

Mounted at /content/drive
Arquivos em: /content/drive/MyDrive/RFAA_P1


In [4]:
import ee
ee.Authenticate()
ee.Initialize(project=ID_PROJETO)
print('Earth Engine conectado:', ID_PROJETO)

EEException: Google Earth Engine API has not been used in project ppgtca-relacoes-fisicas before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/earthengine.googleapis.com/overview?project=ppgtca-relacoes-fisicas then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry.

## Etapa 0 · Locais de estudo
___

Três municípios do norte do Paraná, no planalto basáltico, com clima **Cfa** (Alvares et al., 2013). Eles formam um pequeno gradiente de altitude (Maringá ≈ 540 m, Londrina ≈ 570 m, Apucarana ≈ 820 m), o que permite discutir se os coeficientes calibrados são transferíveis entre locais próximos.

> ⚠️ **Antes de rodar:** confiram no portal do INMET (BDMEP / mapa de estações) o código, as coordenadas e a altitude de cada estação e corrijam a tabela abaixo. O ponto de análise deve ser o da **estação** (critério do P1: estação a até ~10 km). Os valores marcados com `conferir` são aproximados.

In [ ]:
LOCAIS = [
    {'municipio': 'Maringá',   'uf': 'PR', 'estacao_inmet': '83767 (convencional)',
     'lat': -23.4053, 'lon': -51.9328, 'altitude_m': 542.0},
    {'municipio': 'Londrina',  'uf': 'PR', 'estacao_inmet': '83766 (convencional) · conferir',
     'lat': -23.31,   'lon': -51.16,   'altitude_m': 566.0},
    {'municipio': 'Apucarana', 'uf': 'PR', 'estacao_inmet': 'conferir',
     'lat': -23.55,   'lon': -51.46,   'altitude_m': 820.0},
]

ANO_INICIO, ANO_FIM = 2001, 2025
PERIODO = f'{ANO_INICIO}–{ANO_FIM}'
NOMES = [l['municipio'] for l in LOCAIS]
MESES_ABREV = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
CORES_LOCAL = {'Maringá': '#2a78d6', 'Londrina': '#eb6834', 'Apucarana': '#1baf7a'}

tabela1 = pd.DataFrame([{
    'Município': l['municipio'], 'UF': l['uf'], 'Estação INMET': l['estacao_inmet'],
    'Latitude (°)': l['lat'], 'Longitude (°)': l['lon'], 'Altitude (m)': l['altitude_m'],
    'Período': PERIODO, 'N anos': ANO_FIM - ANO_INICIO + 1,
} for l in LOCAIS]).set_index('Município')
tabela1

**Caracterização dos municípios** (continuação da Aula 01): limite municipal (FAO/GAUL), área, altitude média (SRTM 30 m) e chuva anual média (CHIRPS diário).

In [ ]:
# Nomes sem acento, como estão no FAO/GAUL
NOME_GAUL = {'Maringá': 'Maringa', 'Londrina': 'Londrina', 'Apucarana': 'Apucarana'}
municipios = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM1_NAME', 'Parana'))
srtm = ee.Image('USGS/SRTMGL1_003')
chirps_anual = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                .filterDate(f'{ANO_INICIO}-01-01', f'{ANO_FIM + 1}-01-01')
                .sum().divide(ANO_FIM - ANO_INICIO + 1))

AREAS = {}
for l in LOCAIS:
    area = municipios.filter(ee.Filter.eq('ADM2_NAME', NOME_GAUL[l['municipio']]))
    geom = area.geometry()
    AREAS[l['municipio']] = area
    tabela1.loc[l['municipio'], 'Área (km²)'] = round(geom.area().divide(1e6).getInfo(), 1)
    tabela1.loc[l['municipio'], 'Altitude média SRTM (m)'] = round(srtm.reduceRegion(
        ee.Reducer.mean(), geom, 30, maxPixels=1e9).get('elevation').getInfo(), 0)
    tabela1.loc[l['municipio'], 'Chuva anual CHIRPS (mm)'] = round(chirps_anual.reduceRegion(
        ee.Reducer.mean(), geom, 5566, maxPixels=1e9).get('precipitation').getInfo(), 0)

tabela1.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa0_tabela1_locais.csv'))
tabela1

In [ ]:
import geemap

mapa = geemap.Map()
mapa.add_basemap('OpenStreetMap')
for l in LOCAIS:
    mapa.addLayer(AREAS[l['municipio']], {'color': CORES_LOCAL[l['municipio']]}, l['municipio'])
    mapa.addLayer(ee.Geometry.Point([l['lon'], l['lat']]), {'color': 'black'}, f"Ponto {l['municipio']}")
mapa.centerObject(ee.FeatureCollection(list(AREAS.values())).flatten(), 9)
mapa

## Etapa 1 · Extração do BR-DWGD e controle de qualidade
___

**Estratégia de extração.** A coleção `projects/ee-alexandrexavier/assets/BR-DWGD` tem uma imagem por dia. Pedir 25 anos de uma vez (mais de 9 mil imagens) estoura o limite de elementos do `getRegion`. Por isso a extração é feita **ano a ano** (25 chamadas por local) e o resultado é salvo em CSV: se o arquivo já existir, o notebook apenas o lê.

**Fator de escala.** As bandas estão gravadas como inteiros. Cada imagem traz as propriedades `BAND_<banda>_MULT` e `BAND_<banda>_ADD`, e o valor real é `valor = bruto · MULT + ADD`.

In [ ]:
ASSET_XAVIER = 'projects/ee-alexandrexavier/assets/BR-DWGD'
BANDAS = ['pr', 'Tmax', 'Tmin', 'RH', 'u2', 'Rs', 'ET']   # ET = ETo Penman-Monteith do próprio BR-DWGD


def fatores_escala():
    # Lê MULT e ADD de cada banda nas propriedades da primeira imagem
    props = ee.ImageCollection(ASSET_XAVIER).first().toDictionary().getInfo()
    return {b: (props[f'BAND_{b}_MULT'], props[f'BAND_{b}_ADD']) for b in BANDAS}


def extrai_local(local):
    # Extrai a série diária do BR-DWGD no ponto, um ano por vez
    col = ee.ImageCollection(ASSET_XAVIER).select(BANDAS)
    ponto = ee.Geometry.Point([local['lon'], local['lat']])
    fatores = fatores_escala()
    partes = []
    for ano in range(ANO_INICIO, ANO_FIM + 1):
        linhas = col.filterDate(f'{ano}-01-01', f'{ano + 1}-01-01').getRegion(ponto, 10000).getInfo()
        partes.append(pd.DataFrame(linhas[1:], columns=linhas[0]))
        print(f"  {local['municipio']}: {ano} ok", end='\r')
    bruto = pd.concat(partes, ignore_index=True)
    serie = pd.DataFrame({'data': pd.to_datetime(bruto['time'], unit='ms').dt.normalize()})
    for b in BANDAS:
        mult, add = fatores[b]
        serie[b] = bruto[b].astype(float) * mult + add
    return serie.drop_duplicates('data').set_index('data').sort_index()


def carrega_local(local):
    arq = os.path.join(PASTA_DADOS, f"brdwgd_{local['municipio'].lower()}_{ANO_INICIO}_{ANO_FIM}.csv")
    if os.path.exists(arq):
        return pd.read_csv(arq, index_col='data', parse_dates=True)
    serie = extrai_local(local)
    serie.to_csv(arq)
    print(f"\n  salvo em {arq}")
    return serie


BRUTO = {l['municipio']: carrega_local(l) for l in LOCAIS}
for nome, s in BRUTO.items():
    print(f'{nome:<10} {len(s)} dias  ({s.index.min():%d/%m/%Y} a {s.index.max():%d/%m/%Y})')

**Controle de qualidade.** Para cada variável, valores fora de uma faixa plausível viram falha. Há também duas checagens físicas: Tmin > Tmax (as duas temperaturas são descartadas) e Qg > Qo (Qg é descartada). Qo é calculada aqui e reaproveitada na Etapa 2.

In [ ]:
FAIXAS = {'pr': (0, 300), 'Tmax': (-5, 45), 'Tmin': (-10, 35), 'RH': (0, 100),
          'u2': (0, 20), 'Rs': (0, 40), 'ET': (0, 12)}


def astronomicas(df, lat):
    nda = df.index.dayofyear.values
    dec = amp.declinacao_solar(nda)
    Hn = amp.angulo_horario_nascer(lat, dec)
    df['Qo'] = amp.irradiancia_extraterrestre(lat, dec, Hn, amp.fator_correcao_distancia(nda))
    df['N'] = amp.fotoperiodo(Hn)
    return df


def controle_qualidade(serie, lat):
    df = astronomicas(serie.copy(), lat)
    falhas = {}
    for v, (lo, hi) in FAIXAS.items():
        fora = ~df[v].between(lo, hi) & df[v].notna()
        falhas[f'{v} fora da faixa'] = int(fora.sum())
        df.loc[fora, v] = np.nan
    inv_t = df['Tmin'] > df['Tmax']
    inv_r = df['Rs'] > df['Qo']
    falhas['Tmin > Tmax'] = int(inv_t.sum())
    falhas['Qg > Qo'] = int(inv_r.sum())
    df.loc[inv_t, ['Tmax', 'Tmin']] = np.nan
    df.loc[inv_r, 'Rs'] = np.nan
    falhas['Dias com alguma falha'] = int(df[BANDAS].isna().any(axis=1).sum())
    return df, falhas


SERIES, relatorio_qc = {}, {}
for l in LOCAIS:
    SERIES[l['municipio']], relatorio_qc[l['municipio']] = controle_qualidade(BRUTO[l['municipio']], l['lat'])

tabela_qc = pd.DataFrame(relatorio_qc)
tabela_qc.index.name = 'Verificação'
tabela_qc

**Meses secos e úmidos.** Sob Cfa não há estação seca definida. Critério adotado: mês **seco** quando a chuva média mensal é menor que a ETo média mensal (ambas do BR-DWGD).

In [ ]:
CLIMA, MESES_SECOS = {}, {}
for nome, df in SERIES.items():
    df['Tmed'] = (df['Tmax'] + df['Tmin']) / 2
    mensal = df.groupby([df.index.year, df.index.month]).agg(P=('pr', 'sum'), ETo=('ET', 'sum'), T=('Tmed', 'mean'))
    clima = mensal.groupby(level=1).mean()
    clima['seco'] = clima['P'] < clima['ETo']
    CLIMA[nome] = clima
    MESES_SECOS[nome] = clima.index[clima['seco']].tolist()
    k = amp.classificacao_koppen(clima['T'].values, clima['P'].values, LOCAIS[NOMES.index(nome)]['lat'])
    print(f"{nome:<10} Köppen (BR-DWGD): {k['classe']:<4} | meses secos: {[MESES_ABREV[m - 1] for m in MESES_SECOS[nome]]}")

pd.concat({n: c[['P', 'ETo', 'T']].round(1) for n, c in CLIMA.items()}, axis=1)

## Etapa 2 · Variáveis astronômicas e balanço de radiação (FAO-56)
___

| Grandeza | Cálculo |
|---|---|
| es, ea | Tetens pela média de es(Tmax) e es(Tmin); ea = es · UR/100 |
| Qg,cs | (0,75 + 2·10⁻⁵ · z) · Qo (FAO-56, Eq. 37) |
| BOC | Qg · (1 − r) |
| BOL | Stefan-Boltzmann corrigido (FAO-56) |
| Rn | BOC + BOL |

**Albedo.** Usamos **r = 0,23**, o valor da FAO-56 para a grama de referência, porque o Rn calculado aqui serve de referência para a ETo (que no BR-DWGD também é FAO-56). A biblioteca adota 0,25 (gramado) por padrão.

> ⚠️ O Rn de referência é **calculado**, não medido. Fietz & Fisch usaram saldo-radiômetro; essa diferença precisa aparecer no relatório.

In [ ]:
ALBEDO = 0.23

for l in LOCAIS:
    df = SERIES[l['municipio']]
    df['es'] = (amp.es_tetens(df['Tmax']) + amp.es_tetens(df['Tmin'])) / 2
    df['ea'] = amp.ea_umidade(df['es'], df['RH'])
    df['Qgcs'] = (0.75 + 2e-5 * l['altitude_m']) * df['Qo']
    df['BOC'] = amp.boc_saldo(df['Rs'], r=ALBEDO)
    # Na FAO-56, Qg/Qg,cs é limitada a 1 no BOL
    df['BOL'] = amp.bol_saldo(df['Tmax'], df['Tmin'], df['ea'], np.minimum(df['Rs'], df['Qgcs']), df['Qgcs'])
    df['Rn'] = amp.saldo_radiacao(df['BOC'], df['BOL'])

pd.concat({n: SERIES[n][['Qo', 'Rs', 'BOC', 'BOL', 'Rn']].mean() for n in NOMES}, axis=1).round(2)

In [ ]:
estilo = {'Qo': ('0.45', '--', 'o'), 'Rs': ('#f28e2b', '-', 'o'), 'BOC': ('#edc948', '-', 's'),
          'BOL': ('#4e79a7', '-', 'v'), 'Rn': ('#e15759', '-', 'D')}
rotulo = {'Qo': 'Qo', 'Rs': 'Qg', 'BOC': 'BOC', 'BOL': 'BOL', 'Rn': 'Rn'}

fig, eixos = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, nome in zip(eixos, NOMES):
    clim = SERIES[nome].groupby(SERIES[nome].index.month)[list(estilo)].mean()
    for v, (cor, ls, mk) in estilo.items():
        ax.plot(clim.index, clim[v], color=cor, ls=ls, marker=mk, lw=2, label=rotulo[v])
    ax.axhline(0, color='0.6', lw=1)
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(MESES_ABREV, fontsize=8)
    ax.set_title(nome); ax.spines[['top', 'right']].set_visible(False)
eixos[0].set_ylabel('MJ m$^{-2}$ d$^{-1}$')
eixos[1].legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=5)
fig.suptitle(f'Climatologia mensal do balanço de radiação ({PERIODO})')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'etapa2_climatologia_radiacao.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Amplitude térmica mensal: base física dos modelos de Fernandes et al.
fig, eixos = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, nome in zip(eixos, NOMES):
    df = SERIES[nome]
    dados = [(df['Tmax'] - df['Tmin'])[df.index.month == m].dropna() for m in range(1, 13)]
    bp = ax.boxplot(dados, patch_artist=True, widths=0.6, showfliers=False)
    for c in bp['boxes']:
        c.set(facecolor=CORES_LOCAL[nome], alpha=0.55)
    ax.set_xticklabels(MESES_ABREV, fontsize=8); ax.set_title(nome)
    ax.spines[['top', 'right']].set_visible(False)
eixos[0].set_ylabel('Tmax − Tmin (°C)')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'etapa2_boxplot_amplitude.png'), dpi=200, bbox_inches='tight')
plt.show()

## Etapa 3 · Modelos de Qg a partir da temperatura (Fernandes et al., 2018)
___

Variáveis de entrada:

- **ΔTᵢ = Tmaxᵢ − (Tminᵢ + Tminᵢ₊₁)/2** (usa a Tmin do **dia seguinte**; o último dia da série fica sem valor);
- **ΔTavg** = média móvel centrada de 7 dias de ΔTᵢ;
- **Tavg** = (Tmax + Tmin)/2.

| Modelo | Equação | Parâmetros |
|---|---|---|
| HG (Hargreaves) | Qg = b · Qo · √ΔTᵢ + c | b, c |
| BC (Bristow-Campbell) | Qg = 0,75 · [1 − exp(−b · ΔTᵢᶜ / ΔTavg)] · Qo | b, c |
| CD (Donatelli-Campbell) | Qg = 0,75 · [1 − exp(−b · f(Tavg) · ΔTᵢ² · exp(Tmin/Tnc))] · Qo | b, Tnc |
| DB (Donatelli-Bellocchi) | Qg = 0,75 · [1 + f(NDA)] · [1 − exp(−b · ΔTᵢ² / ΔTavg)] · Qo | b, C1, C2 |
| DCBB | Qg = 0,75 · [1 + f(NDA)] · [1 − exp(−b · ΔTᵢ² · exp(Tmin/Tnc) / ΔTavg)] · Qo | b, C1, C2, Tnc |

com f(Tavg) = 0,017 · exp[exp(−0,053 · Tavg)] e f(NDA) = C1 · [sen(NDA · C2 · π/180) + cos(NDA · f₃ · π/180)], f₃ = 1 − 1,9·C3 + 3,83·C3², C3 = C2 − int(C2).

> O HG de Fernandes et al. tem **dois** parâmetros calibrados (b e c), e não o k fixo de `amp.etp_hargreaves_samani`.

In [ ]:
for df in SERIES.values():
    df['dTi'] = df['Tmax'] - (df['Tmin'] + df['Tmin'].shift(-1)) / 2
    df['dTavg'] = df['dTi'].rolling(7, center=True, min_periods=7).mean()
    df['NDA'] = df.index.dayofyear


def f_nda(nda, C1, C2):
    C3 = C2 - np.floor(C2)
    f3 = 1 - 1.9 * C3 + 3.83 * C3 ** 2
    return C1 * (np.sin(nda * C2 * np.pi / 180) + np.cos(nda * f3 * np.pi / 180))


def m_HG(p, d):
    b, c = p
    return b * d['Qo'] * np.sqrt(np.clip(d['dTi'], 0, None)) + c


def m_BC(p, d):
    b, c = p
    return 0.75 * (1 - np.exp(-b * np.clip(d['dTi'], 0, None) ** c / d['dTavg'])) * d['Qo']


def m_CD(p, d):
    b, Tnc = p
    fT = 0.017 * np.exp(np.exp(-0.053 * d['Tavg']))
    return 0.75 * (1 - np.exp(-b * fT * d['dTi'] ** 2 * np.exp(d['Tmin'] / Tnc))) * d['Qo']


def m_DB(p, d):
    b, C1, C2 = p
    return 0.75 * (1 + f_nda(d['NDA'], C1, C2)) * (1 - np.exp(-b * d['dTi'] ** 2 / d['dTavg'])) * d['Qo']


def m_DCBB(p, d):
    b, C1, C2, Tnc = p
    return (0.75 * (1 + f_nda(d['NDA'], C1, C2))
            * (1 - np.exp(-b * d['dTi'] ** 2 * np.exp(d['Tmin'] / Tnc) / d['dTavg'])) * d['Qo'])


# chute inicial e limites de cada parâmetro
MODELOS_QG = {
    'HG':   dict(f=m_HG,   par=['b', 'c'],               x0=[0.16, 0.0],          lim=([0, -10], [1, 10])),
    'BC':   dict(f=m_BC,   par=['b', 'c'],               x0=[0.5, 1.5],           lim=([1e-3, 0.1], [20, 4])),
    'CD':   dict(f=m_CD,   par=['b', 'Tnc'],             x0=[0.1, 30.0],          lim=([1e-4, 1], [5, 200])),
    'DB':   dict(f=m_DB,   par=['b', 'C1', 'C2'],        x0=[0.1, 0.05, 1.5],     lim=([1e-4, -0.5, 0.01], [5, 0.5, 3])),
    'DCBB': dict(f=m_DCBB, par=['b', 'C1', 'C2', 'Tnc'], x0=[0.1, 0.05, 1.5, 30], lim=([1e-4, -0.5, 0.01, 1], [5, 0.5, 3, 200])),
}
ORDEM_QG = list(MODELOS_QG)

**Métricas próprias** (exigência do P1). Oᵢ observado, Pᵢ estimado.

In [ ]:
def _arr(o, p):
    o, p = np.asarray(o, float), np.asarray(p, float)
    ok = np.isfinite(o) & np.isfinite(p)
    return o[ok], p[ok]


def r_pearson(o, p):
    o, p = _arr(o, p)
    return np.sum((o - o.mean()) * (p - p.mean())) / np.sqrt(np.sum((o - o.mean()) ** 2) * np.sum((p - p.mean()) ** 2))


def rmse(o, p):
    o, p = _arr(o, p)
    return np.sqrt(np.mean((p - o) ** 2))


def rrmse(o, p):
    o, p = _arr(o, p)
    return 100 * rmse(o, p) / o.mean()


def mae(o, p):
    o, p = _arr(o, p)
    return np.mean(np.abs(p - o))


def vies(o, p):
    o, p = _arr(o, p)
    return np.mean(p - o)


def ef_nash(o, p):
    o, p = _arr(o, p)
    return 1 - np.sum((p - o) ** 2) / np.sum((o - o.mean()) ** 2)


def d_willmott(o, p):
    o, p = _arr(o, p)
    return 1 - np.sum((p - o) ** 2) / np.sum((np.abs(p - o.mean()) + np.abs(o - o.mean())) ** 2)


def c_camargo(o, p):
    return r_pearson(o, p) * d_willmott(o, p)


def classe_c(c):
    # Camargo & Sentelhas (1997)
    for lim, nome in [(0.85, 'Ótimo'), (0.76, 'Muito bom'), (0.66, 'Bom'), (0.61, 'Mediano'), (0.51, 'Sofrível'), (0.41, 'Mau')]:
        if c > lim:
            return nome
    return 'Péssimo'


def metricas(o, p):
    r = r_pearson(o, p)
    return {'RMSE': rmse(o, p), 'RRMSE_%': rrmse(o, p), 'MAE': mae(o, p), 'Viés': vies(o, p),
            'EF': ef_nash(o, p), 'R2': r ** 2, 'd': d_willmott(o, p), 'c': c_camargo(o, p),
            'Desempenho': classe_c(r * d_willmott(o, p))}

**Calibração.** Mínimos quadrados não lineares (`scipy.optimize.least_squares`) nos anos ímpares. DB e DCBB são sensíveis ao chute inicial; por isso cada modelo é ajustado a partir de vários chutes (o chute da tabela + sorteios dentro dos limites) e fica o de menor erro. A semente é fixa para o resultado ser reprodutível.

In [ ]:
def calibra(func, dados, alvo, x0, lim, n_chutes=1, seed=42):
    lo, hi = np.array(lim[0], float), np.array(lim[1], float)
    rng = np.random.default_rng(seed)
    chutes = [np.array(x0, float)] + [rng.uniform(lo, hi) for _ in range(n_chutes - 1)]
    melhor, custo_min = None, np.inf
    for x in chutes:
        try:
            res = least_squares(lambda p: func(p, dados) - alvo, np.clip(x, lo, hi), bounds=(lo, hi), max_nfev=5000)
        except Exception:
            continue
        custo = np.sum(res.fun ** 2)
        if np.isfinite(custo) and custo < custo_min:
            melhor, custo_min = res.x, custo
    return melhor


BASE_QG, PAR_QG, EST_QG = {}, {}, {}
for nome in NOMES:
    base = SERIES[nome][['Qo', 'dTi', 'dTavg', 'Tmed', 'Tmin', 'NDA', 'Rs']].rename(columns={'Tmed': 'Tavg'}).dropna()
    base = base[base['dTavg'] > 0]
    base['grupo'] = np.where(base.index.year % 2 == 1, 'Calibração', 'Validação')
    cal = base[base['grupo'] == 'Calibração']
    BASE_QG[nome], PAR_QG[nome], EST_QG[nome] = base, {}, {}
    for m, info in MODELOS_QG.items():
        n_ch = 40 if m in ('DB', 'DCBB') else 10
        PAR_QG[nome][m] = calibra(info['f'], cal, cal['Rs'].values, info['x0'], info['lim'], n_ch)
        EST_QG[nome][m] = pd.Series(np.asarray(info['f'](PAR_QG[nome][m], base)), index=base.index)
    print(f"{nome}: {len(cal)} dias de calibração, {len(base) - len(cal)} de validação")

In [ ]:
# Tabela 2: parâmetros calibrados
tabela2 = pd.DataFrame([{'Local': n, 'Modelo': m, **dict(zip(MODELOS_QG[m]['par'], PAR_QG[n][m]))}
                        for n in NOMES for m in ORDEM_QG]).set_index(['Modelo', 'Local'])
tabela2.round(5)

In [ ]:
# Tabela 3: métricas na calibração e na validação
reg = []
for n in NOMES:
    b = BASE_QG[n]
    for m in ORDEM_QG:
        for g in ('Calibração', 'Validação'):
            sel = b['grupo'] == g
            reg.append({'Modelo': m, 'Local': n, 'Conjunto': g, **metricas(b.loc[sel, 'Rs'], EST_QG[n][m][sel])})
tabela3 = pd.DataFrame(reg).set_index(['Modelo', 'Local', 'Conjunto'])
tabela3.round(3)

**d e c por período (seco, úmido, todo) em escala diária e mensal**, na série completa.

In [ ]:
reg = []
for n in NOMES:
    b = BASE_QG[n]
    seco = b.index.month.isin(MESES_SECOS[n])
    for m in ORDEM_QG:
        for per, sel in [('Todo', np.ones(len(b), bool)), ('Seco', seco), ('Úmido', ~seco)]:
            o, p = b['Rs'][sel], EST_QG[n][m][sel]
            mens = pd.DataFrame({'o': o, 'p': p}).groupby([o.index.year, o.index.month]).mean()
            reg.append({'Modelo': m, 'Local': n, 'Período': per,
                        'd_diário': d_willmott(o, p), 'c_diário': c_camargo(o, p), 'classe_diário': classe_c(c_camargo(o, p)),
                        'd_mensal': d_willmott(mens['o'], mens['p']), 'c_mensal': c_camargo(mens['o'], mens['p']),
                        'classe_mensal': classe_c(c_camargo(mens['o'], mens['p']))})
tabela_dc = pd.DataFrame(reg).set_index(['Modelo', 'Local', 'Período'])
tabela_dc.round(3)

In [ ]:
fig, eixos = plt.subplots(len(NOMES), len(ORDEM_QG), figsize=(17, 10), sharex=True, sharey=True)
for i, n in enumerate(NOMES):
    b = BASE_QG[n]
    val = b['grupo'] == 'Validação'
    for j, m in enumerate(ORDEM_QG):
        ax = eixos[i, j]
        ax.scatter(b.loc[val, 'Rs'], EST_QG[n][m][val], s=3, alpha=0.2, color=CORES_LOCAL[n], edgecolors='none')
        ax.plot([0, 35], [0, 35], color='0.4', ls='--', lw=1)
        met = tabela3.loc[(m, n, 'Validação')]
        ax.text(0.04, 0.96, f"RMSE={met['RMSE']:.2f}\nd={met['d']:.3f}", transform=ax.transAxes, va='top', fontsize=8)
        ax.set_xlim(0, 35); ax.set_ylim(0, 35)
        if i == 0:
            ax.set_title(m)
        if j == 0:
            ax.set_ylabel(f'{n}\nQg estimada')
        if i == len(NOMES) - 1:
            ax.set_xlabel('Qg BR-DWGD')
fig.suptitle('Qg estimada × Qg do BR-DWGD na validação (anos pares), MJ m$^{-2}$ d$^{-1}$')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'etapa3_dispersao_qg.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Melhor modelo de cada local: maior EF na validação
MELHOR_QG = {n: tabela3.xs((n, 'Validação'), level=('Local', 'Conjunto'))['EF'].idxmax() for n in NOMES}
print('Melhor modelo de Qg (EF na validação):', MELHOR_QG)

tabela2.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa3_tabela2_parametros_qg.csv'))
tabela3.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa3_tabela3_metricas_qg.csv'))
tabela_dc.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa3_d_c_periodos.csv'))

**Os coeficientes são transferíveis?** Cada local é estimado com os parâmetros calibrados nos outros dois. Se a perda de desempenho for pequena, um único conjunto de coeficientes serve para a região.

In [ ]:
reg = []
for m in ORDEM_QG:
    for n_cal in NOMES:
        for n_uso in NOMES:
            b = BASE_QG[n_uso]
            val = b['grupo'] == 'Validação'
            est = MODELOS_QG[m]['f'](PAR_QG[n_cal][m], b[val])
            reg.append({'Modelo': m, 'Calibrado em': n_cal, 'Aplicado em': n_uso, 'RMSE': rmse(b.loc[val, 'Rs'], est)})
transf = pd.DataFrame(reg).pivot_table(index=['Modelo', 'Calibrado em'], columns='Aplicado em', values='RMSE')[NOMES]
transf.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa3_transferencia_rmse.csv'))
transf.round(3)

## Etapa 4 · Modelos de Rn e Priestley-Taylor (Fietz & Fisch, 2009)
___

Quatro modelos de Rn, calibrados contra o Rn FAO-56 da Etapa 2 (anos ímpares):

| Modelo | Equação |
|---|---|
| (1) | Rn = α · Tmax + β · Tmin |
| (2) | Rn = α · (Tmax − Tmin)^β · Qo |
| (3) | Rn = α · Qg |
| (4) | Rn = α · Qg + β · Qo |

**Priestley-Taylor:** ET₀ = α_PT · W · Rn / λ, com α_PT = 1,26, λ = 2,45 MJ kg⁻¹ e G ≈ 0 na escala diária. Fietz & Fisch usam aproximações lineares de W = Δ/(Δ+γ) em função da temperatura, que comparamos com o W exato.

In [ ]:
MODELOS_RN = {
    '(1)': dict(f=lambda p, d: p[0] * d['Tmax'] + p[1] * d['Tmin'],                                       par=['α', 'β'], x0=[0.3, 0.0]),
    '(2)': dict(f=lambda p, d: p[0] * np.clip(d['Tmax'] - d['Tmin'], 1e-6, None) ** p[1] * d['Qo'],      par=['α', 'β'], x0=[0.05, 0.7]),
    '(3)': dict(f=lambda p, d: p[0] * d['Rs'],                                                           par=['α'],      x0=[0.6]),
    '(4)': dict(f=lambda p, d: p[0] * d['Rs'] + p[1] * d['Qo'],                                          par=['α', 'β'], x0=[0.5, 0.0]),
}
ORDEM_RN = list(MODELOS_RN)

BASE_RN, PAR_RN, EST_RN = {}, {}, {}
for n in NOMES:
    b = SERIES[n][['Tmax', 'Tmin', 'Tmed', 'Qo', 'Rs', 'Rn', 'ET']].dropna().copy()
    b['grupo'] = np.where(b.index.year % 2 == 1, 'Calibração', 'Validação')
    cal = b[b['grupo'] == 'Calibração']
    BASE_RN[n], PAR_RN[n], EST_RN[n] = b, {}, {}
    for m, info in MODELOS_RN.items():
        PAR_RN[n][m] = least_squares(lambda p: info['f'](p, cal) - cal['Rn'].values, info['x0']).x
        EST_RN[n][m] = pd.Series(np.asarray(info['f'](PAR_RN[n][m], b)), index=b.index)

tabela_par_rn = pd.DataFrame([{'Modelo': m, 'Local': n, **dict(zip(MODELOS_RN[m]['par'], PAR_RN[n][m]))}
                              for m in ORDEM_RN for n in NOMES]).set_index(['Modelo', 'Local'])
tabela_par_rn.round(4)

In [ ]:
reg = []
for n in NOMES:
    b = BASE_RN[n]
    for m in ORDEM_RN:
        for g in ('Calibração', 'Validação'):
            sel = b['grupo'] == g
            est = EST_RN[n][m][sel]
            reg.append({'Modelo': m, 'Local': n, 'Conjunto': g, 'Rn médio': est.mean(),
                        'CV_%': 100 * est.std() / est.mean(), **metricas(b.loc[sel, 'Rn'], est)})
tabela_rn = pd.DataFrame(reg).set_index(['Modelo', 'Local', 'Conjunto'])
tabela_rn.round(3)

In [ ]:
fig, eixos = plt.subplots(len(NOMES), len(ORDEM_RN), figsize=(14, 10), sharex=True, sharey=True)
for i, n in enumerate(NOMES):
    b = BASE_RN[n]
    val = b['grupo'] == 'Validação'
    for j, m in enumerate(ORDEM_RN):
        ax = eixos[i, j]
        ax.scatter(b.loc[val, 'Rn'], EST_RN[n][m][val], s=3, alpha=0.2, color=CORES_LOCAL[n], edgecolors='none')
        ax.plot([-2, 22], [-2, 22], color='0.4', ls='--', lw=1)
        met = tabela_rn.loc[(m, n, 'Validação')]
        ax.text(0.04, 0.96, f"R²={met['R2']:.3f}\nc={met['c']:.3f}", transform=ax.transAxes, va='top', fontsize=8)
        if i == 0:
            ax.set_title(f'Modelo {m}')
        if j == 0:
            ax.set_ylabel(f'{n}\nRn estimado')
        if i == len(NOMES) - 1:
            ax.set_xlabel('Rn FAO-56')
fig.suptitle('Rn estimado × Rn FAO-56 na validação, MJ m$^{-2}$ d$^{-1}$')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'etapa4_dispersao_rn.png'), dpi=200, bbox_inches='tight')
plt.show()

**Razão Rn/Qg mensal.** O modelo (3) supõe Rn proporcional a Qg. Como o BOL quase não diminui no inverno enquanto Qg cai, a razão deve cair do verão para o inverno.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for n in NOMES:
    df = SERIES[n].dropna(subset=['Rn', 'Rs'])
    razao = df.groupby(df.index.month).apply(lambda d: d['Rn'].sum() / d['Rs'].sum())
    ax.plot(range(1, 13), razao.values, marker='o', lw=2, color=CORES_LOCAL[n], label=f'{n} (anual {razao.mean():.2f})')
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MESES_ABREV)
ax.set_ylabel('Rn / Qg'); ax.set_title(f'Razão Rn/Qg mensal ({PERIODO})')
ax.spines[['top', 'right']].set_visible(False); ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'etapa4_razao_rn_qg.png'), dpi=200, bbox_inches='tight')
plt.show()

**Fator W.** Aproximação linear de Fietz & Fisch × W exato = Δ/(Δ+γ), com Δ pela derivada de Tetens e γ pela pressão atmosférica da altitude (FAO-56).

In [ ]:
ALFA_PT, LAMBDA = 1.26, 2.45


def W_linear(T):
    T = np.asarray(T, float)
    return np.where(T <= 16, 0.407 + 0.0145 * T, 0.483 + 0.01 * T)


def W_exato(T, z):
    T = np.asarray(T, float)
    delta = 4098 * 0.6108 * np.exp(17.27 * T / (T + 237.3)) / (T + 237.3) ** 2
    P = 101.3 * ((293 - 0.0065 * z) / 293) ** 5.26
    gama = 0.665e-3 * P
    return delta / (delta + gama)


T = np.linspace(5, 32, 100)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(T, W_linear(T), color='k', lw=2, label='Linear (Fietz & Fisch)')
for l in LOCAIS:
    ax.plot(T, W_exato(T, l['altitude_m']), ls='--', color=CORES_LOCAL[l['municipio']], label=f"Exato, {l['municipio']} ({l['altitude_m']:.0f} m)")
ax.set_xlabel('Temperatura média (°C)'); ax.set_ylabel('W'); ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'etapa4_fator_w.png'), dpi=200, bbox_inches='tight')
plt.show()

for l in LOCAIS:
    df = SERIES[l['municipio']]
    df['W'] = W_linear(df['Tmed'])
    df['W_exato'] = W_exato(df['Tmed'], l['altitude_m'])
    df['ETo_PT'] = ALFA_PT * df['W'] * df['Rn'] / LAMBDA
    print(f"{l['municipio']:<10} diferença média W linear − W exato: {(df['W'] - df['W_exato']).mean():+.4f}")

**Equação local.** Se Rn ≈ α₃ · Qg (modelo 3), Priestley-Taylor vira **ET₀ = k · W · Qg**, com k = α_PT · α₃ / λ.

In [ ]:
K_LOCAL = {}
reg = []
for n in NOMES:
    df = SERIES[n]
    K_LOCAL[n] = ALFA_PT * PAR_RN[n]['(3)'][0] / LAMBDA
    df['ETo_local'] = K_LOCAL[n] * df['W'] * df['Rs']
    print(f'{n:<10} ET0 = {K_LOCAL[n]:.4f} · W · Qg')
    for nome, col in [('Priestley-Taylor (Rn FAO-56)', 'ETo_PT'), ('Equação local k·W·Qg', 'ETo_local')]:
        reg.append({'Local': n, 'Método': nome, **metricas(df['ET'], df[col])})
tabela_pt = pd.DataFrame(reg).set_index(['Local', 'Método'])
tabela_pt.round(3)

**α_PT sazonal.** Recalibrado por período (regressão pela origem de ETo PM contra W·Rn/λ). Valores acima de 1,26 indicam demanda atmosférica maior que a explicada só pela energia (advecção, ar seco).

In [ ]:
reg = []
for n in NOMES:
    b = SERIES[n][['W', 'Rn', 'ET']].dropna()
    seco = b.index.month.isin(MESES_SECOS[n])
    X = (b['W'] * b['Rn'] / LAMBDA).values
    y = b['ET'].values
    linha = {'Local': n}
    for per, sel in [('Todo', np.ones(len(b), bool)), ('Seco', seco), ('Úmido', ~seco)]:
        linha[per] = np.sum(y[sel] * X[sel]) / np.sum(X[sel] ** 2) if sel.any() else np.nan
    reg.append(linha)
tabela_alfa = pd.DataFrame(reg).set_index('Local')

tabela_par_rn.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa4_parametros_rn.csv'))
tabela_rn.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa4_metricas_rn.csv'))
tabela_pt.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa4_pt_vs_pm.csv'))
tabela_alfa.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa4_alfa_pt.csv'))
tabela_alfa.round(3)

## Etapa 5 · Integração: temperatura → Qg → Rn → ETo
___

Três estimativas de ETo, todas comparadas com a ETo Penman-Monteith do BR-DWGD:

| Cenário | Qg | Rn | ETo |
|---|---|---|---|
| (a) | BR-DWGD | FAO-56 (Etapa 2) | Priestley-Taylor |
| (b) | **melhor modelo de temperatura** (Etapa 3) | modelo (3) · Qg estimada | Priestley-Taylor |
| (c) | n/a | n/a | Hargreaves-Samani (só temperatura) |

Comparar (a) com (b) isola o erro que **nasce na estimativa de Qg**. O erro de (a) é o que já vem do Priestley-Taylor, da aproximação de W e do Rn calculado.

> As métricas da Etapa 5 usam só os **anos pares** (validação), para que nenhum coeficiente seja avaliado nos dados em que foi calibrado.

In [ ]:
ETO = {}
reg = []
for n in NOMES:
    df = SERIES[n]
    b = BASE_QG[n].join(df[['Tmax', 'W', 'ET', 'ETo_PT']], how='inner').dropna()
    b = b[b['grupo'] == 'Validação']
    m = MELHOR_QG[n]
    qg_est = MODELOS_QG[m]['f'](PAR_QG[n][m], b)
    rn_est = PAR_RN[n]['(3)'][0] * qg_est
    e = pd.DataFrame({
        'PM': b['ET'],
        '(a) PT, Qg BR-DWGD': b['ETo_PT'],
        f'(b) PT, Qg por temperatura ({m})': ALFA_PT * b['W'] * rn_est / LAMBDA,
        '(c) Hargreaves-Samani': amp.etp_hargreaves_samani(b['Qo'], b['Tmax'], b['Tmin'], b['Tavg']),
    }, index=b.index)
    ETO[n] = e
    for col in e.columns[1:]:
        reg.append({'Local': n, 'Cenário': col[:3], 'Descrição': col[4:], **metricas(e['PM'], e[col])})
tabela5 = pd.DataFrame(reg).set_index(['Local', 'Cenário'])
tabela5.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa5_integracao.csv'))
tabela5.round(3)

In [ ]:
# Em que elo nasce o erro? Decomposição do erro quadrático médio (RMSE²) de (b).
# Parcela negativa: Qg estimada compensou parte do erro do restante da cadeia (erros de sinais opostos).
reg = []
for n in NOMES:
    e = ETO[n]
    col_b = [c for c in e.columns if c.startswith('(b)')][0]
    mse_a = rmse(e['PM'], e['(a) PT, Qg BR-DWGD']) ** 2
    mse_b = rmse(e['PM'], e[col_b]) ** 2
    reg.append({'Local': n, 'RMSE (a)': np.sqrt(mse_a), 'RMSE (b)': np.sqrt(mse_b),
                'RMSE (c)': rmse(e['PM'], e['(c) Hargreaves-Samani']),
                'Parcela do erro de (b) vinda de Qg (%)': 100 * (mse_b - mse_a) / mse_b})
tabela_elo = pd.DataFrame(reg).set_index('Local')
tabela_elo.to_csv(os.path.join(PASTA_RESULTADOS, 'etapa5_origem_erro.csv'))
tabela_elo.round(3)

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for ax, n in zip(eixos, NOMES):
    e = ETO[n]
    for k, col in enumerate(e.columns[1:]):
        r = e.groupby(e.index.month).apply(lambda d: rmse(d['PM'], d[col]))
        ax.bar(np.arange(12) + (k - 1) * 0.27, r.values, width=0.27, label=col[:3], color=['#2a78d6', '#eb6834', '#7f7f7f'][k])
    ax.set_xticks(range(12)); ax.set_xticklabels(MESES_ABREV, fontsize=8)
    ax.set_title(n); ax.spines[['top', 'right']].set_visible(False)
eixos[0].set_ylabel('RMSE da ETo (mm d$^{-1}$)')
eixos[1].legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3)
fig.suptitle('RMSE mensal da ETo em relação à Penman-Monteith (validação)')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'etapa5_rmse_mensal.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, eixos = plt.subplots(len(NOMES), 3, figsize=(13, 12), sharex=True, sharey=True)
for i, n in enumerate(NOMES):
    e = ETO[n]
    for j, col in enumerate(e.columns[1:]):
        ax = eixos[i, j]
        ax.scatter(e['PM'], e[col], s=3, alpha=0.2, color=CORES_LOCAL[n], edgecolors='none')
        ax.plot([0, 9], [0, 9], color='0.4', ls='--', lw=1)
        met = tabela5.loc[(n, col[:3])]
        ax.text(0.04, 0.96, f"RMSE={met['RMSE']:.2f}\nc={met['c']:.3f}", transform=ax.transAxes, va='top', fontsize=8)
        if i == 0:
            ax.set_title(col, fontsize=9)
        if j == 0:
            ax.set_ylabel(f'{n}\nETo estimada')
        if i == len(NOMES) - 1:
            ax.set_xlabel('ETo Penman-Monteith')
plt.tight_layout()
plt.savefig(os.path.join(PASTA_FIGURAS, 'etapa5_dispersao_eto.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Base final de cada local, com todas as variáveis calculadas
for n in NOMES:
    SERIES[n].to_csv(os.path.join(PASTA_DADOS, f'serie_final_{n.lower()}_{ANO_INICIO}_{ANO_FIM}.csv'))
print('Resultados em:', os.path.abspath(PASTA_RESULTADOS))
print('Figuras em:   ', os.path.abspath(PASTA_FIGURAS))

## Tabelas para o relatório
___

Gera `resultados/tabelas_relatorio.md` com as tabelas do relatório (`relatorio/relatorio_p1.md`) já formatadas em Markdown, com vírgula decimal. Basta copiar cada bloco para o lugar marcado no relatório.

In [ ]:
def tabela_md(df, casas=3):
    # Converte um DataFrame em tabela Markdown com vírgula decimal (sem depender do pacote tabulate)
    df = df.reset_index()

    def fmt(v):
        if isinstance(v, (float, np.floating)):
            return '' if np.isnan(v) else f'{v:.{casas}f}'.replace('.', ',')
        return str(v)

    linhas = ['| ' + ' | '.join(map(str, df.columns)) + ' |', '|' + '---|' * len(df.columns)]
    linhas += ['| ' + ' | '.join(fmt(v) for v in linha) + ' |' for linha in df.itertuples(index=False)]
    return '\n'.join(linhas)


clima_tab = pd.concat({n: c[['P', 'ETo', 'T']] for n, c in CLIMA.items()}, axis=1)
clima_tab.columns = [f'{a} {b}' for a, b in clima_tab.columns]
clima_tab.index = MESES_ABREV
clima_tab.index.name = 'Mês'
val3 = tabela3.xs('Validação', level='Conjunto')[['RMSE', 'RRMSE_%', 'MAE', 'Viés', 'EF', 'd', 'c', 'Desempenho']]
blocos = {
    'Tabela 1. Locais de estudo': (tabela1, 2),
    'Tabela 2. Controle de qualidade (dias descartados)': (tabela_qc, 0),
    'Tabela 3. Climatologia mensal: chuva (P, mm), ETo (mm) e temperatura média (T, °C)':
        (clima_tab, 1),
    'Tabela 4. Parâmetros calibrados dos modelos de Qg': (tabela2, 4),
    'Tabela 5. Desempenho dos modelos de Qg na validação (anos pares)': (val3, 3),
    'Tabela 6. Índice c por período (série completa, escala diária)':
        (tabela_dc['c_diário'].unstack('Período')[['Todo', 'Seco', 'Úmido']], 3),
    'Tabela 7. RMSE (MJ m⁻² d⁻¹) com coeficientes transferidos entre locais': (transf, 3),
    'Tabela 8. Parâmetros dos modelos de Rn': (tabela_par_rn, 4),
    'Tabela 9. Desempenho dos modelos de Rn na validação':
        (tabela_rn.xs('Validação', level='Conjunto')[['Rn médio', 'RMSE', 'EF', 'R2', 'd', 'c', 'Desempenho']], 3),
    'Tabela 10. αPT recalibrado por período': (tabela_alfa, 3),
    'Tabela 11. ETo estimada × Penman-Monteith (validação)':
        (tabela5[['Descrição', 'RMSE', 'Viés', 'EF', 'R2', 'd', 'c', 'Desempenho']], 3),
    'Tabela 12. Origem do erro da cadeia só com temperatura': (tabela_elo, 3),
}
texto = [f'# Tabelas do relatório P1 ({PERIODO})\n']
for titulo, (df, casas) in blocos.items():
    texto += [f'**{titulo}**\n', tabela_md(df, casas), '']
texto += ['**Equação local (Etapa 4):** ' + '; '.join(f'{n}: ET₀ = {K_LOCAL[n]:.4f}·W·Qg'.replace('.', ',') for n in NOMES),
          '', '**Melhor modelo de Qg (EF na validação):** ' + ', '.join(f'{n}: {m}' for n, m in MELHOR_QG.items())]

arq_md = os.path.join(PASTA_RESULTADOS, 'tabelas_relatorio.md')
with open(arq_md, 'w', encoding='utf-8') as f:
    f.write('\n'.join(texto))
print('Tabelas salvas em:', arq_md)

## Para o relatório
___

Pontos para discutir com base nas tabelas acima:

1. **Qualidade da base:** quantos dias foram descartados em cada local e por quê (tabela do controle de qualidade). Lembrar que Qg do BR-DWGD é interpolada, não medida.
2. **Qg por temperatura:** qual modelo foi melhor em cada local, se o ranking se repete nos três e se o desempenho cai nos meses secos ou úmidos.
3. **Transferência:** os coeficientes calibrados em uma cidade funcionam nas outras? Apucarana, mais alta, se comporta diferente?
4. **Rn:** a razão Rn/Qg muda ao longo do ano; o que isso diz sobre o modelo (3)? Comparar os α com Fietz & Fisch (Dourados).
5. **Priestley-Taylor:** o α_PT recalibrado fica acima ou abaixo de 1,26 nos meses secos? Por quê?
6. **Pergunta central:** com só termômetro, quanto erro chega à ETo (cenário b) e quanto disso nasce em Qg (tabela da origem do erro)? Vale a pena em relação ao Hargreaves-Samani (cenário c)?

### Referências

ALLEN, R. G.; PEREIRA, L. S.; RAES, D.; SMITH, M. **Crop evapotranspiration**: guidelines for computing crop water requirements. Rome: FAO, 1998. (Irrigation and Drainage Paper, 56).

ALVARES, C. A. et al. Köppen's climate classification map for Brazil. **Meteorologische Zeitschrift**, v. 22, n. 6, p. 711-728, 2013.

CAMARGO, A. P.; SENTELHAS, P. C. Avaliação do desempenho de diferentes métodos de estimativa da evapotranspiração potencial no Estado de São Paulo, Brasil. **Revista Brasileira de Agrometeorologia**, v. 5, n. 1, p. 89-97, 1997.

FERNANDES, D. S.; HEINEMANN, A. B.; AMORIM, A. O.; PAZ, R. L. F. Estimativa da radiação solar global com base em observações de temperatura para o estado de Goiás. **Revista Brasileira de Meteorologia**, v. 33, n. 3, p. 558-566, 2018.

FIETZ, C. R.; FISCH, G. F. Avaliação de modelos de estimativa do saldo de radiação e do método de Priestley-Taylor para a região de Dourados, MS. **Revista Brasileira de Engenharia Agrícola e Ambiental**, v. 13, n. 4, p. 449-453, 2009.

XAVIER, A. C.; SCANLON, B. R.; KING, C. W.; ALVES, A. I. New improved Brazilian daily weather gridded data (1961–2020). **International Journal of Climatology**, v. 42, n. 16, p. 8390-8404, 2022.